# GraphGenerator on ZINC

This notebook demonstrates the two-stage `GraphGenerator`: first generate a new interpretation graph, then instantiate base molecules conditionally from nearby ZINC examples.

In [1]:
from pathlib import Path
import runpy

BOOTSTRAP_CANDIDATES = (
    "notebooks/_bootstrap.py",
    "abstractgraph/notebooks/_bootstrap.py",
    "abstractgraph-ml/notebooks/_bootstrap.py",
    "abstractgraph-generative/notebooks/_bootstrap.py",
    "abstractgraph-graphicalizer/notebooks/_bootstrap.py",
)

_bootstrap_path = next(
    (
        candidate / relative
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for relative in BOOTSTRAP_CANDIDATES
        if (candidate / relative).exists()
    ),
    None,
)
if _bootstrap_path is None:
    raise FileNotFoundError("Could not locate ecosystem notebooks/_bootstrap.py")

_bootstrap = runpy.run_path(str(_bootstrap_path))
repo_root = _bootstrap["repo_root"]
workspace_root = _bootstrap["workspace_root"]


In [2]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

from collections import Counter

from nsppk import NSPPK
from sklearn.ensemble import RandomForestClassifier

from abstractgraph.display import display, display_decomposition_graph, display_mappings
from abstractgraph.graphs import graph_to_abstract_graph
from abstractgraph.operators import *
from abstractgraph_graphicalizer.chem import ZINCLoader, draw_molecules as display_graphs
from abstractgraph_ml.estimators import GraphEstimator
from abstractgraph_generative.conditional import ConditionalAutoregressiveGenerator
from abstractgraph_generative.edge_generator import EdgeGenerator
from abstractgraph_generative.graph_generator import GraphGenerator
from abstractgraph_ml.feasibility import FeasibilityEstimator, FeasibilityEstimatorFeatureCannotExist



In [3]:
def draw(graph, decomposition_function, *, nbits=11, label_mode="operator_hash", size=(12, 6), n_elements_per_row=8):
    ag = graph_to_abstract_graph(
        graph,
        decomposition_function=decomposition_function,
        nbits=nbits,
        label_mode=label_mode,
    )
    display(ag, size=size)
    display_mappings(ag, n_elements_per_row=n_elements_per_row)
    return ag

---

In [4]:
loader = ZINCLoader(on_error="skip")

dataset_name = "zinc_250k"
size = 3000
min_num_nodes = 30
max_num_nodes = 40

graphs, metadata = loader.load(
    dataset_name,
    limit=size,
    min_node_count=min_num_nodes,
    max_node_count=max_num_nodes,
)

print(f"dataset: {dataset_name}")
print(f"n_graphs: {len(graphs)}")
print(f"node_range: [{min_num_nodes}, {max_num_nodes}]")


dataset: zinc_250k
n_graphs: 3000
node_range: [30, 40]


In [5]:
label_mode = "histogram_values" #label_mode: str = "operator_hash" (default) or "histogram" or "histogram_values" for AbstractGraph node labeling.

nbits = 14

cycle_tree = add(
    compose(name("cycle"), cycle()),
    compose(name("tree"), tree()),
)
decomposition_function = compose(intersection_edges(), cycle_tree)

edge_vectorizer = NSPPK(radius=1, distance=3, connector=1, nbits=12, dense=True, parallel=True)
edge_graph_estimator = GraphEstimator(
    transformer=edge_vectorizer,
    estimator=RandomForestClassifier(
        n_estimators=80,
        random_state=0,
        n_jobs=-1,
        class_weight="balanced_subsample",
    ),
)

#------------------------------------------------------------------------------------------------------------------------------------------------------------------------
feasibility_kwargs = dict(
    nbits=19,
    parallel=True,
    backend="loky",
    n_jobs=-1,
)
partial_feasibility_estimators = [
    FeasibilityEstimatorFeatureCannotExist(
        decomposition_function=compose(neighborhood(radius=2), unlabel()),
        **feasibility_kwargs,
    ),
    FeasibilityEstimatorFeatureCannotExist(
        decomposition_function=neighborhood(radius=1),
        **feasibility_kwargs,
    ),
]
partial_feasibility_estimator = FeasibilityEstimator(partial_feasibility_estimators)

final_feasibility_estimators = [
    FeasibilityEstimatorFeatureCannotExist(
        decomposition_function=compose(neighborhood(radius=2), unlabel()),
        **feasibility_kwargs,
    ),
    FeasibilityEstimatorFeatureCannotExist(
        decomposition_function=neighborhood(radius=1),
        **feasibility_kwargs,
    ),
]
final_feasibility_estimator = FeasibilityEstimator(final_feasibility_estimators)
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------

edge_generator = EdgeGenerator(
    partial_feasibility_estimator=partial_feasibility_estimator,
    final_feasibility_estimator=final_feasibility_estimator,
    graph_estimator=edge_graph_estimator,
    n_negative_per_positive=3,
    n_replicates=2,
    beam_size=3,
    max_restarts=2,
    fit_n_jobs=-1,
    fit_backend="loky",
    seed=0,
)

graph_vectorizer = NSPPK(radius=1, distance=4, connector=1, nbits=14, dense=True, parallel=True)
conditional_generator = ConditionalAutoregressiveGenerator(
    decomposition_function=decomposition_function,
    nbits=nbits,
    label_mode=label_mode,
    base_cut_radius=0,
    interpretation_cut_radius=1,
    context_vectorizer=graph_vectorizer,
    n_jobs=1,
)

generator = GraphGenerator(
    edge_generator=edge_generator,
    conditional_generator=conditional_generator,
    decomposition_function=decomposition_function,
    nbits=nbits,
    label_mode=label_mode,
    interpretation_neighbor_vectorizer=NSPPK(radius=1, distance=3, connector=1, nbits=12, dense=True, parallel=True),
    seed=None,
    debug=True,
    require_new_interpretation_graph=True,
    max_same_interpretation_retries=3,
)


In [6]:
generator.store(graphs)

In [ ]:
n_samples = 4
n_instances_per_sample = 3
generated_graphs = generator.sample(
    n_samples=n_samples,
    n_interpretation_neighbors=10,
    n_conditional_neighbors=10,
    n_instances_per_sample=n_instances_per_sample,
    interpretation_edge_removal_size=1,  # 0 bypasses edge generation; 1 removes all interpretation edges before regrowth.
    random_state=None,
    conditional_generate_kwargs=dict(
        random_state=None,
        max_backtracks=2000,
        max_attempts_per_sample=6,
        require_signature_coverage=True,
    ),
)

generated_groups = [
    generated_graphs[i : i + n_instances_per_sample]
    for i in range(0, len(generated_graphs), n_instances_per_sample)
]

print(f"generated molecules: {len(generated_graphs)}")
print("attempted seed indices:", generator.last_sampled_indices_)
print("successful seed indices:", generator.last_successful_sampled_indices_)
print("successful interpretation targets:", len(generator.last_generated_interpretation_graphs_))
if not generated_graphs:
    print("No molecules generated; inspect warnings and try a larger neighborhood or dataset slice.")


[graph-generator edge] seed_idx=403 n_neighbors=10 neighbor_indices=[800, 1846, 1683, 2323, 1709, 1425, 2113, 2234, 1352, 1669] neighbor_distances=[33.5261, 38.223, 38.3145, 38.3406, 38.3797, 38.6394, 38.7814, 38.8201, 38.8973, 38.8973]
[fit] partial_feasibility_graphs=109 positives=124 negatives=312 dataset=436 partial_time=0m 1.4s
[fit] final_feasibility_graphs=10 final_time=0m 0.2s
[fit] lookahead_envelope_stages=7 lookahead_examples=124 lookahead_rejection_model=lognormal_tail
[fit] graph_estimator_graphs=436 positive_labels=124 negative_labels=312 time=0m 3.8s
[graph 0] start start_edges=0 target_edges=7 remaining_edges=7
[graph 0] search_phase=1/3 beam_limit=3
[graph 0] remaining_edges=6
search_phase=1/3 depth=1 step_time=0m 3.3s eta=0m 19.7s
tried=56 generated=56 partial_infeasible=22 partial_feasible=34 lookahead_infeasible=6 viable=28 retained=3
best_score=0.812 best_selection_score=0.745 best_repulsion=0.332
repulsion_lambda=0.200 beam_limit=3
[graph 0] remaining_edges=5
sear

/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/abstractgraph-ecosystem/repos/abstractgraph-generative/src/abstractgraph_generative/graph_generator.py:280: RuntimeWarning: Edge stage failed to generate an interpretation graph; skipping seed.
  self._generate_interpretation_graph_with_retries(


[graph-generator edge] seed_idx=2651 n_neighbors=10 neighbor_indices=[2932, 595, 516, 1774, 2844, 1223, 2545, 1767, 2010, 1492] neighbor_distances=[26.4386, 28.0357, 28.5307, 30.8221, 28.0357, 28.178, 28.2135, 28.6356, 28.775, 29.0]
[fit] partial_feasibility_graphs=84 positives=100 negatives=240 dataset=340 partial_time=0m 0.3s
[fit] final_feasibility_graphs=10 final_time=0m 0.2s
[fit] lookahead_envelope_stages=6 lookahead_examples=100 lookahead_rejection_model=lognormal_tail
[fit] graph_estimator_graphs=340 positive_labels=100 negative_labels=240 time=0m 3.9s
[graph 0] start start_edges=0 target_edges=6 remaining_edges=6
[graph 0] search_phase=1/3 beam_limit=3
[graph 0] remaining_edges=5
search_phase=1/3 depth=1 step_time=0m 4.0s eta=0m 20.0s
tried=63 generated=63 partial_infeasible=54 partial_feasible=9 viable=9 retained=2
best_score=0.723 best_selection_score=0.663 best_repulsion=0.300
repulsion_lambda=0.200 beam_limit=3
[graph 0] remaining_edges=4
search_phase=1/3 depth=2 step_time

/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/abstractgraph-ecosystem/repos/abstractgraph-generative/src/abstractgraph_generative/graph_generator.py:280: RuntimeWarning: Edge stage failed to generate an interpretation graph; skipping seed.
  self._generate_interpretation_graph_with_retries(


[graph-generator edge] seed_idx=453 n_neighbors=10 neighbor_indices=[1470, 756, 1155, 2563, 2597, 882, 921, 744, 1285, 2428] neighbor_distances=[30.7246, 33.3167, 31.5595, 33.5112, 33.5559, 33.8821, 34.1614, 34.641, 34.7419, 34.9285]
[fit] partial_feasibility_graphs=134 positives=146 negatives=378 dataset=524 partial_time=0m 0.5s
[fit] final_feasibility_graphs=10 final_time=0m 0.2s
[fit] lookahead_envelope_stages=9 lookahead_examples=146 lookahead_rejection_model=lognormal_tail
[fit] graph_estimator_graphs=524 positive_labels=146 negative_labels=378 time=0m 4.2s
[graph 0] start start_edges=0 target_edges=8 remaining_edges=8
[graph 0] search_phase=1/3 beam_limit=3
[graph 0] remaining_edges=7
search_phase=1/3 depth=1 step_time=0m 6.9s eta=0m 48.5s
tried=72 generated=72 partial_infeasible=22 partial_feasible=50 viable=50 retained=3
best_score=0.863 best_selection_score=0.799 best_repulsion=0.319
repulsion_lambda=0.200 beam_limit=3
[graph 0] remaining_edges=6
search_phase=1/3 depth=2 step_

In [ ]:
if not generated_graphs:
    print("No generated molecules to display.")
else:
    for sample_idx, (
        seed_idx,
        seed_graph,
        seed_interpretation_graph,
        generated_interpretation_graph,
        generated_instances,
    ) in enumerate(
        zip(
            generator.last_successful_sampled_indices_,
            generator.last_seed_graphs_,
            generator.last_seed_interpretation_graphs_,
            generator.last_generated_interpretation_graphs_,
            generated_groups,
        )
    ):
        print("=" * 120)
        print(f"sample {sample_idx} | seed index {seed_idx}")

        print("seed molecule")
        display_graphs([seed_graph], n_graphs_per_line=1)

        print("seed interpretation graph")
        display([seed_interpretation_graph], size=(5, 4))

        print("generated interpretation graph")
        display([generated_interpretation_graph], size=(5, 4))

        print(f"generated molecule instances ({len(generated_instances)})")
        display_graphs(generated_instances, n_graphs_per_line=n_instances_per_sample)
        for generated_instance in generated_instances:
            draw(generated_instance, decomposition_function=decomposition_function, nbits=nbits, label_mode=label_mode)
